In [ ]:
from guppylang.std.builtins import array, nat, qubit
from guppylang import guppy
from guppylang.std.quantum import measure_array, x, h, measure, cx, cz

from hugr.hugr.render import RenderConfig

n = guppy.nat_var("n")
c = guppy.nat_var("c")

@guppy.unitary
class foo:

    @guppy
    def __call__(q1: array[qubit, 2]) -> None:
        h(q1[0])

    @guppy
    def call_daggered(q1: array[qubit, 2]) -> None:
        x(q1[0])

    @guppy
    def call_controlled(q1: array[qubit, 2], _controls: array[qubit, n]) -> None:
        cz(_controls[0], q1[0])

    @guppy
    def call_ctrl_daggered(q1: array[qubit, n], _controls: array[qubit, c]) -> None:
        cx(_controls[0], q1[0])
        

@guppy
def main() -> None:
    q1 = array(qubit(), qubit())
    foo(q1)
    measure_array(q1)

main.check()
# main.compile().modules[0] #.render_dot(RenderConfig(max_node_label_length=50, max_metadata_length=None)) #.view()


In [ ]:
@guppy.protocol
class MyProto:
    @guppy.require
    def foo(self: "MyProto", xaaaa: int) -> str: ...

@guppy.struct(frozen=True)
class MyType:
    @guppy
    def foo(self: "MyType", t: int) -> str:
        return "something"

@guppy
def bar[M: MyProto](a: M) -> str:
    return a.foo(42)

# Internally desugared this should be equivalent to `bar`.
@guppy
def baz(a: MyProto) -> str:
    return a.foo(42)

@guppy
def main() -> None:
    mt = MyType()
    bar(mt)
    baz(mt)

main.compile()

In [ ]:
from collections.abc import Callable

from guppylang.decorator import guppy
from guppylang.std.array import array

from guppylang.std.builtins import (
    Controllable,
    Daggerable,
    Unitary,
    control,
    dagger,
    owned,
    power,
    result,
)
from guppylang.std.num import nat
from guppylang.std.quantum import (
    angle,
    cx,
    discard,
    h,
    measure,
    qubit,
    rx,
    discard_array,
)
from guppylang_internals.metadata.common import (
    CONTROLLED_KEY,
    CTRL_DAGGERED_KEY,
    DAGGERED_KEY,
)
from hugr.ops import FuncDefn

def test_unitary_class_functions_are_regular_functions():
    @guppy.unitary
    class foo:
        @guppy
        def helper(q: qubit) -> None:
            h(q)

        @guppy
        def __call__(q: qubit) -> None:
            helper(q)

        @guppy
        def call_daggered(q: qubit) -> None:
            helper(q)

        @guppy
        def call_controlled(q: qubit, _controls: array[qubit, 1]) -> None:
            helper(q)
            helper(_controls[0])

        @guppy
        def call_ctrl_daggered(q: qubit, _controls: array[qubit, 1]) -> None:
            helper(q)
            helper(_controls[0])

    @guppy
    def main() -> None:
        q = qubit()
        foo(q)
        result("r", measure(q).read())

    return main

test_unitary_class_functions_are_regular_functions().emulator(n_qubits=1).with_shots(100).run().collated_counts()

In [1]:

from guppylang.decorator import guppy
from guppylang.std.array import array

from guppylang.std.builtins import (
    control,
    dagger,
)
from guppylang.std.num import nat
from guppylang.std.quantum import ( cx,
    h, x,
    measure,
    qubit)
import guppylang
guppylang.enable_experimental_features()

def test_unitary_class_functions_are_regular_functions():
    @guppy.unitary
    class foo:
        @guppy
        def __call__(q: qubit) -> None:
            h(q)
            c = qubit()
            measure(c)

        @guppy
        def call_daggered(q: qubit) -> None:
            x(q)

        # @guppy
        # def call_controlled[c: nat](q: qubit, _controls: array[qubit, c]) -> None:
        #     cx(q, _controls[0])

        # @guppy
        # def call_ctrl_daggered(q: qubit, _controls: array[qubit, 1]) -> None:
        #     cx(q, _controls[0])

    @guppy
    def main() -> None:
        q = qubit()
        c = qubit()
        with control(c):
            foo(q)
        with dagger:
            foo(q)
        with control(c), dagger:
            foo(q)
        measure(q)
        measure(c)

    return main

test_unitary_class_functions_are_regular_functions().check()

Error: Control constraint violation (at <In[1]>:42:12)
   | 
40 |         c = qubit()
41 |         with control(c):
42 |             foo(q)
   |             ^^^^^^ This function cannot be called in a controllable context

Help: The `@guppy.unitary` function `foo` is missing a custom implementation.
Consider implementing the custom method `call_controlled`

Guppy compilation failed due to 1 previous error


In [1]:

from guppylang.decorator import guppy
from guppylang.std.array import array

from guppylang.std.builtins import (
    control,
    dagger,
)
from guppylang.std.num import nat
from guppylang.std.quantum import ( cx,
    h, x,
    measure,
    qubit)
import guppylang
guppylang.enable_experimental_features()

def test_unitary_class_functions_are_regular_functions():
    @guppy.unitary
    class foo:
        @guppy
        def __call__(q: qubit) -> None:
            h(q)
            c = qubit()
            measure(c)

        @guppy
        def call_daggered(q: qubit) -> None:
            x(q)

        # @guppy
        # def call_controlled[c: nat](q: qubit, _controls: array[qubit, c]) -> None:
        #     cx(q, _controls[0])

        # @guppy
        # def call_ctrl_daggered(q: qubit, _controls: array[qubit, 1]) -> None:
        #     cx(q, _controls[0])

    @guppy(unitary=True)
    def main(q: qubit) -> None:
        foo(q)
        
    return main

test_unitary_class_functions_are_regular_functions().check()

Error: Control constraint violation (at <In[1]>:39:8)
   | 
37 |     @guppy(unitary=True)
38 |     def main(q: qubit) -> None:
39 |         foo(q)
   |         ^^^^^^ This function cannot be called in a controllable context

Help: The `@guppy.unitary` function `foo` is missing a custom implementation.
Consider implementing the custom method `call_controlled`

Guppy compilation failed due to 1 previous error


In [ ]:
from guppylang import guppy, qubit
from guppylang.std.builtins import owned


@guppy.declare(unitary=True)
def uni_discard(q: qubit @owned) -> None: ...


@guppy(unitary=True)
def test() -> None:
    p = qubit()
    uni_discard(p)

test.compile()
